# palatenet milestone

starting small: turn 2,000 yelp restaurant reviews into a graph, then try one
mean-aggregation layer. users and restaurants are the two node types.
this runs on cpu. no training yet.

In [1]:
import json
import platform
from pathlib import Path

import torch
import torch_geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv

seed = 42
max_reviews = 2_000
device = torch.device("cpu")
# works from the project root or this milestone folder
project_dir = Path.cwd()
if not (project_dir / "data/yelp_academic_dataset_business.json").is_file():
    project_dir = project_dir.parent
data_dir = project_dir / "data"
business_path = data_dir / "yelp_academic_dataset_business.json"
review_path = data_dir / "yelp_academic_dataset_review.json"

# keeping this small enough to run locally
torch.manual_seed(seed)
assert max_reviews > 0
for path in (business_path, review_path):
    if not path.is_file():
        raise FileNotFoundError(f"missing data file: {path.resolve()}")

print("python:", platform.python_version())
print("torch:", torch.__version__, "pyg:", torch_geometric.__version__)
print("device:", device, "seed:", seed)

/Users/tenzintsultrim/Class/CS224W/palatenet/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/tenzintsultrim/Class/CS224W/palatenet/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


python: 3.11.5
torch: 2.15.0 pyg: 2.8.1
device: cpu seed: 42


## grab the restaurant ids

these are json lines files, so each line is a separate record. only keep
businesses with the exact `Restaurants` category, not every food-related business.

In [2]:
def json_rows(path):
    # the review file is huge, so don't load it all at once
    with path.open(encoding="utf-8") as file:
        for line_num, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"bad json in {path}, line {line_num}") from error
            if not isinstance(row, dict):
                raise ValueError(f"expected an object in {path}, line {line_num}")
            yield row


def valid_id(value):
    return isinstance(value, str) and bool(value.strip())


restaurant_ids = set()
for business in json_rows(business_path):
    business_id = business.get("business_id")
    categories = business.get("categories")
    if not valid_id(business_id) or not isinstance(categories, str):
        continue

    categories = {category.strip() for category in categories.split(",")}
    if "Restaurants" in categories:
        restaurant_ids.add(business_id)

## get a small review sample

keep the first valid review for each user–restaurant pair and stop at 2,000.
this follows file order, which isn't necessarily date order. it's a development
sample, not a representative sample or a train/test split.

In [3]:
reviews = []
seen_pairs = set()

for review in json_rows(review_path):
    user_id = review.get("user_id")
    restaurant_id = review.get("business_id")
    stars = review.get("stars")

    if not valid_id(user_id) or not valid_id(restaurant_id):
        continue
    if restaurant_id not in restaurant_ids:
        continue
    if isinstance(stars, bool) or not isinstance(stars, (int, float)):
        continue
    if stars not in (1, 2, 3, 4, 5):
        continue

    # keep the first valid record if the same pair shows up again
    pair = (user_id, restaurant_id)
    if pair in seen_pairs:
        continue
    seen_pairs.add(pair)
    reviews.append((user_id, restaurant_id, float(stars)))
    if len(reviews) >= max_reviews:
        break

if not reviews:
    raise ValueError("no valid restaurant reviews found; check the input files")
assert len(seen_pairs) == len(reviews)
assert all(stars in (1, 2, 3, 4, 5) for _, _, stars in reviews)
print("real yelp review pairs:", len(reviews))

real yelp review pairs: 2000


## turn the ids into indices

users and restaurants get separate index spaces in first-seen order.
user 0 and restaurant 0 are different nodes. keep the ratings in the same
order as the edges.

In [4]:
user_map = {}
restaurant_map = {}
user_indices = []
restaurant_indices = []
stars_list = []

for user_id, restaurant_id, stars in reviews:
    if user_id not in user_map:
        user_map[user_id] = len(user_map)
    if restaurant_id not in restaurant_map:
        restaurant_map[restaurant_id] = len(restaurant_map)

    user_indices.append(user_map[user_id])
    restaurant_indices.append(restaurant_map[restaurant_id])
    stars_list.append(stars)

num_users = len(user_map)
num_restaurants = len(restaurant_map)
num_reviews = len(reviews)
edge_index = torch.tensor([user_indices, restaurant_indices], dtype=torch.long, device=device)
ratings = torch.tensor(stars_list, dtype=torch.float32, device=device).reshape(-1, 1)

# swap the rows so restaurants can send messages back to users
reverse_edges = edge_index.flip(0).contiguous()

## build the pyg graph

random features are just placeholders to try out message passing. they aren't
user preferences or restaurant attributes. adding reverse edges gives us
two directed edges per review, not twice as many reviews.

In [5]:
torch.manual_seed(seed)
graph = HeteroData()
graph["user"].x = torch.randn(num_users, 8, device=device)
graph["restaurant"].x = torch.randn(num_restaurants, 8, device=device)

forward = ("user", "reviews", "restaurant")
reverse = ("restaurant", "rev_reviews", "user")
graph[forward].edge_index = edge_index
graph[forward].edge_attr = ratings
graph[reverse].edge_index = reverse_edges

# a few checks before passing anything through the layer
assert len({(user, restaurant) for user, restaurant, _ in reviews}) == num_reviews
assert edge_index.shape == (2, num_reviews)
assert edge_index.dtype == torch.long
assert 0 <= edge_index[0].min() <= edge_index[0].max() < num_users
assert 0 <= edge_index[1].min() <= edge_index[1].max() < num_restaurants
assert ratings.shape == (num_reviews, 1)
assert ratings.dtype == torch.float32
assert torch.equal(reverse_edges, edge_index.flip(0))
assert graph["user"].x.shape == (num_users, 8)
assert graph["restaurant"].x.shape == (num_restaurants, 8)
assert graph.validate(raise_on_error=True)

for i, (user_id, restaurant_id, stars) in enumerate(reviews):
    assert edge_index[0, i].item() == user_map[user_id]
    assert edge_index[1, i].item() == restaurant_map[restaurant_id]
    assert ratings[i, 0].item() == stars

print("users:", num_users)
print("restaurants:", num_restaurants)
print("reviews:", num_reviews)
print("forward edges:", graph[forward].num_edges)
print("reverse edges:", graph[reverse].num_edges)
print("total directed edges:", graph.num_edges)
print(graph)

users: 1970
restaurants: 1140
reviews: 2000
forward edges: 2000
reverse edges: 2000
total directed edges: 4000
HeteroData(
  user={ x=[1970, 8] },
  restaurant={ x=[1140, 8] },
  (user, reviews, restaurant)={
    edge_index=[2, 2000],
    edge_attr=[2000, 1],
  },
  (restaurant, rev_reviews, user)={ edge_index=[2, 2000] }
)


## try one mean-aggregation layer

each node gets a transformed mean of its neighbors' vectors plus a transformation
of its own vector. use a separate SAGEConv for each direction, going from 8 to 16
features. both directions read the original inputs so this is one simultaneous layer.

the stored stars aren't passed into SAGEConv. this doesn't distinguish good reviews
from bad ones yet. `eval()` and `no_grad()` don't mean we've trained or measured accuracy.

In [6]:
torch.manual_seed(seed)
to_restaurants = SAGEConv((8, 8), 16, aggr="mean").to(device)
to_users = SAGEConv((8, 8), 16, aggr="mean").to(device)
to_restaurants.eval()
to_users.eval()

user_x = graph["user"].x
restaurant_x = graph["restaurant"].x

# don't overwrite the inputs between the two directions
with torch.no_grad():
    restaurant_out = to_restaurants((user_x, restaurant_x), edge_index)
    user_out = to_users((restaurant_x, user_x), reverse_edges)

assert restaurant_out.shape == (num_restaurants, 16)
assert user_out.shape == (num_users, 16)
assert torch.isfinite(restaurant_out).all()
assert torch.isfinite(user_out).all()
assert user_out.device.type == restaurant_out.device.type == "cpu"
print("restaurant output:", tuple(restaurant_out.shape))
print("user output:", tuple(user_out.shape))
print("checks passed. real yelp data, cpu only, no training yet.")

restaurant output: (1140, 16)
user output: (1970, 16)
checks passed. real yelp data, cpu only, no training yet.


## where this leaves things

the data-to-graph step and one forward pass work. the outputs are untrained node
vectors, not predicted ratings, so there's no accuracy result to report.

the rating predictor in `palatenet_rating_predictor.ipynb` builds on this example.
it uses a chronological split and keeps target reviews out of the input graph.
comparing mean, max, and standard deviation is still future work.

references: [HeteroData](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.data.HeteroData.html),
[SAGEConv](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.SAGEConv.html)